# 📊 COVID-19 Forecasting with SimpleFeedForward

## 🎯 Project Goal

**Fast baseline forecast of US COVID-19 cases using SimpleFeedForward**

This notebook demonstrates:
- Quick baseline forecasting (< 2 minutes total runtime)
- Simple model for trend following
- Comparison baseline for more complex models

**When to use SimpleFeedForward**:
- ✅ Need quick results
- ✅ Data shows simple trends
- ✅ Limited computational resources
- ✅ Baseline for comparison

**Data**: Same COVID-19 data as DeepAR example  
**Runtime**: ~1-2 minutes  
**Difficulty**: Beginner

In [ ]:
import sys
sys.path.append('.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from utils.load_data_utils import DataLoader
from utils.preprocess_data_utils import aggregate_to_national, extract_national_mobility, merge_all_data
from utils.gluonts_utils import create_gluonts_dataset, verify_dataset, prepare_train_test_split
from utils.evaluation_utils import calculate_metrics, print_metrics

from gluonts.torch.model.simple_feedforward import SimpleFeedForwardEstimator
from gluonts.evaluation import make_evaluation_predictions

print("✓ Setup complete!"
)

---

## 📥 Load and Preprocess Data

Same data pipeline as Deep AR - loading and preprocessing COVID data.

In [ ]:
print("📥 Loading COVID-19 data...")

loader = DataLoader(data_dir="data")
cases_df = loader.load_cases()
deaths_df = loader.load_deaths()
mobility_df = loader.load_mobility()
vaccine_df = loader.load_vaccines()

print("✓ Data loaded")

print("\n🔧 Preprocessing...")
national_cases = aggregate_to_national(cases_df, data_type='cases')
national_deaths = aggregate_to_national(deaths_df, data_type='deaths')
national_mobility = extract_national_mobility(mobility_df)

merged_df = merge_all_data(national_cases, national_deaths, national_mobility, vaccine_df)

print(f"✓ Preprocessed: {len(merged_df)} days of data")
print(f"  Date range: {merged_df['Date'].min().date()} to {merged_df['Date'].max().date()}")
merged_df.head()

---

## 🔧 Prepare for SimpleFeedForward

**Key Difference from DeepAR**: We'll use fewer features since SimpleFeedForward is simpler.

We'll focus on the core signal (cases) without too many exogenous features.

In [ ]:
TARGET = 'Daily_Cases_MA7'

# Use only key features (SimpleFeedForward works better with fewer)
selected_features = [
    'Daily_Deaths_MA7',
    'retail_and_recreation_percent_change_from_baseline',
    'workplaces_percent_change_from_baseline'
]

print(f"Target: {TARGET}")
print(f"\nUsing {len(selected_features)} key features:")
for feat in selected_features:
    print(f"  • {feat}")

# Split data
train_df, test_df = prepare_train_test_split(merged_df, test_size=14, target_column=TARGET)

# Convert to GluonTS
train_ds = create_gluonts_dataset(train_df, TARGET, 'D', 14, past_feat_columns=selected_features)
test_ds = create_gluonts_dataset(test_df, TARGET, 'D', 14, past_feat_columns=selected_features)

verify_dataset(train_ds, "Train")
verify_dataset(test_ds, "Test")

---

## 🤖 Train SimpleFeedForward

**Configuration**:
- Small context window (21 days)
- Simple network ([40] neurons)
- Fast training (10 epochs)

**Expected time**: ~30-60 seconds

In [ ]:
print("🏋️ Training SimpleFeedForward model...")
print("=" * 60)

# SimpleFeedForward has simpler parameters than DeepAR
estimator = SimpleFeedForwardEstimator(
    prediction_length=14,      # Forecast 14 days ahead
    context_length=60,         # Use 60 days of history for better context
    hidden_dimensions=[40],    # One hidden layer with 40 neurons
    num_feat_dynamic_real=len(data['features']) if data['features'] else 0,
    trainer=dict(
        max_epochs=10,         # Quick training
        learning_rate=0.001    # Learning rate
    )
)

print("\n📚 Training... (this is FAST!)")
print("  SimpleFeedForward trains very quickly...")
predictor = estimator.train(train_ds)

print("=" * 60)
print("✓ Training complete!")
print("  SimpleFeedForward trained in seconds!")

---

## 🔮 Generate Forecasts

In [ ]:
print("🔮 Generating forecasts...")

forecast_it, ts_it = make_evaluation_predictions(test_ds, predictor, num_samples=100)
forecasts = list(forecast_it)
ground_truths = list(ts_it)

forecast = forecasts[0]
actual = ground_truths[0]

print("✓ Forecasts generated!")
print(f"\nMean forecast: {forecast.mean.mean():,.0f} daily cases")
print(f"Range: {forecast.mean.min():,.0f} to {forecast.mean.max():,.0f}")

---

## 📊 Evaluate Performance

In [ ]:
forecast_period = 14
actual_values = actual[-forecast_period:]
forecast_values = forecast.mean

def calculate_metrics(pred, true):
    # Convert to numpy arrays to avoid pandas dimension issues
    pred_array = np.array(pred)
    true_array = np.array(true)
    errors = pred_array - true_array
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    mape = np.mean(np.abs(errors / true_array)) * 100
    return {'mae': mae, 'rmse': rmse, 'mape': mape}

metrics = calculate_metrics(forecast_values, actual_values)

print("\n📊 SimpleFeedForward Performance:")
print("=" * 60)
print(f"MAE:  {metrics['mae']:>10,.0f} cases")
print(f"RMSE: {metrics['rmse']:>10,.0f} cases")
print(f"MAPE: {metrics['mape']:>10.2f} %")
print("=" * 60)

if metrics['mape'] < 15:
    print("\n✓ Good performance for a simple model!")
elif metrics['mape'] < 25:
    print("\n✓ Reasonable baseline performance")
else:
    print("\n⚠️  Complex patterns may need DeepAR")

---

## 📈 Visualize Results

In [ ]:
plt.figure(figsize=(14, 6))

# Context
train_context = train_df.tail(60)
plt.plot(train_context['Date'], train_context[TARGET],
         label='Historical', color='steelblue', linewidth=2, alpha=0.8)

# Forecast dates
last_date = train_df['Date'].iloc[-1]
forecast_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=14, freq='D')

# Actual vs Forecast
plt.plot(forecast_dates, actual_values,
         label='Actual', color='orange', linewidth=3, marker='o', markersize=8)

plt.plot(forecast_dates, forecast_values,
         label='SimpleFeedForward', color='green', linewidth=3, 
         marker='s', markersize=7, linestyle='--')

# Confidence
plt.fill_between(forecast_dates, forecast.quantile(0.1), forecast.quantile(0.9),
                  alpha=0.2, color='green', label='80% Confidence')

plt.title('COVID-19 Forecasting: SimpleFeedForward (Fast Baseline)', 
          fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Daily Cases (7-Day MA)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig('feedforward_covid_forecast.png', dpi=150, bbox_inches='tight')
print("✓ Plot saved")
plt.show()

---

## 🎓 Summary

**SimpleFeedForward Results**:
- ✅ Fast training (< 1 minute)
- ✅ Simple baseline established
- ✅ Reasonable trend following

**When SimpleFeedForward Works Well**:
- Stable trends
- Short-term forecasts
- Quick prototyping
- Computational constraints

**When to Use DeepAR Instead**:
- Complex seasonal patterns
- Long-term dependencies
- Multiple waves/regime changes
- Better accuracy needed

**Next**: Compare with `GluonTS_DeepAR.example.ipynb` to see the difference!